# Managing Resources in Azure ML

## Notebook Setup

In [15]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path, os
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: c:\Users\dmika\DEV\Projects-local\dp100-learn
Changed working directory to: c:\Users\dmika\DEV\Projects-local\dp100-learn


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Creating a Workspace

This section covers the steps to create an Azure Machine Learning workspace. You will need to have the right access to be able to create resources in your Azure subscription. We will demonstrate the full process but note that in many cases, workspaces are created by administrators and shared with data scientists and developers.

In [ ]:
raise NotImplementedError("This section is a template that you should fill before using.")

### Create a Resource Group

Here we will create a dedicated resource group for our Azure ML workspace. A resource group is a logical container that holds related Azure resources.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import ResourceManagementClient
from utils.consts import AZUREML_RESOURCE_GROUP as resource_group, AZUREML_RESOURCE_LOCATION as resource_group_location

# Create resource management client
# resource_group = resource_group.replace("dev", "test")
credential = DefaultAzureCredential()
resource_client = ResourceManagementClient(credential, subscription_id)

try:
    # Try to get existing resource group
    rg_result = resource_client.resource_groups.get(resource_group)
    print(f"Resource group '{rg_result.name}' already exists in region '{rg_result.location}'")
except ResourceNotFoundError:
    # Create resource group if it doesn't exist
    rg_result = resource_client.resource_groups.create_or_update(
        resource_group,
        {"location": resource_group_location}
    )
    print(f"Provisioned resource group '{rg_result.name}' in the {rg_result.location} region")


# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

Provisioned resource group 'azure-ml-test-rg' in the westeurope region


### (Optional) Create a Storage Account

When you create an Azure ML workspace, a storage account is automatically created for you. However, in some scenarios, you might want to create and manage your own storage account separately or you already have an existing one you wish to use.

In [2]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.storage import StorageManagementClient
from utils.consts import AZUREML_RESOURCE_GROUP as resource_group, AZUREML_RESOURCE_LOCATION as resource_group_location, AZUREML_STORAGE_ACCOUNT_NAME as storage_account_name


credential = DefaultAzureCredential()
storage_client = StorageManagementClient(credential, subscription_id)

try:
    # Try to get existing storage account
    storage_account = storage_client.storage_accounts.get_properties(
        resource_group_name=resource_group,
        account_name=storage_account_name
    )
    print(f"Storage account already exists: {storage_account.name}")
except ResourceNotFoundError:
    # If not found, create new storage account
    print("Creating storage account...")
    poller = storage_client.storage_accounts.begin_create(
        resource_group_name=resource_group,
        account_name=storage_account_name,
        parameters={
            "location": resource_group_location,
            "sku": {"name": "Standard_LRS"},
            "kind": "StorageV2",
            "enable_https_traffic_only": True
        }
    )
    storage_account = poller.result()
    print(f"Storage account created: {storage_account.name}")

# Extract the resource ID
storage_resource_id = storage_account.id
print(f"Storage Resource ID: {storage_resource_id}")

Creating storage account...
Storage account created: azuremldevstorage10099
Storage Resource ID: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/providers/Microsoft.Storage/storageAccounts/azuremldevstorage10099


### Create a Workspace

In [ ]:
from azure.core.exceptions import ResourceNotFoundError
from azure.ai.ml.entities import Workspace
from utils.consts import AZUREML_RESOURCE_GROUP as resource_group, AZUREML_RESOURCE_LOCATION as resource_group_location, AZUREML_STORAGE_ACCOUNT_NAME as storage_account_name, AZUREML_WORKSPACE_NAME as workspace_name


# resource_group = resource_group.replace("dev", "test")
# workspace_name = workspace_name.replace("dev", "test")
credential = DefaultAzureCredential()
try:
    # Try to get existing workspace
    ws = ml_client.workspaces.get(workspace_name)
    print(f"AML workspace already exists: {ws.name}")
except (ResourceNotFoundError, TypeError):
    ml_client = MLClient(
        credential=credential,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
    )
    # Create new AML workspace
    # NOTE: if creating a new workspace fails with workspace name already exists, it might be that ML workspace was not permanently deleted. To do so you need to go to Azure Portal -> Azure Machine Learning and delete it from there.
    ws = Workspace(
        name=workspace_name,
        location=resource_group_location,
        # storage_account=storage_resource_id,
    )
    print("Creating AML workspace...")
    ml_client.workspaces.begin_create(ws).result()
    ml_client = MLClient(
        credential=credential,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        workspace_name=workspace_name,
    )

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Creating AML workspace...


The deployment request azure-ml-test-ws-2875293 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2Fa1267753-4c98-48c1-a8e9-9c7169202ffd%2FresourceGroups%2Fazure-ml-test-rg%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Fazure-ml-test-ws-2875293
Creating Log Analytics Workspace: (azuremltlogalytid56b8a6e  ) ...  Done (21s)
Creating AzureML Workspace: (azure-ml-test-ws  ) ..  Done (17s)
Creating Application Insights: (azuremltinsightsccac0b6d  )  Done (23s)
Creating Storage Account: (azuremltstorage365188dfa  )  Done (26s)
Creating Key Vault: (azuremltkeyvault5b46c61d  )  Done (23s)
Total time : 45s

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while alr

### Creating a workspace with existing resources

In [4]:
from utils.consts import AZUREML_SUBSCRIPTION_ID as subscription_id, AZUREML_RESOURCE_GROUP as resource_group, AZUREML_STORAGE_ACCOUNT_NAME as storage_account_name

test_resource_group = resource_group.replace("dev", "test")
test_workspace_name = workspace_name.replace("dev", "test")

credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=test_resource_group,
    workspace_name=test_workspace_name
)
azureml_test_ws = ml_client.workspaces.get(test_workspace_name)
azureml_test_ws

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Workspace({'kind': 'default', 'print_as_yaml': False, 'discovery_url': 'https://westeurope.api.azureml.ms/discovery', 'mlflow_tracking_uri': 'azureml://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-test-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-test-ws', 'workspace_id': '61447b22-2422-430e-914e-cd2baeea7368', 'feature_store_settings': None, 'name': 'azure-ml-test-ws', 'description': 'azure-ml-test-ws', 'tags': {'createdByToolkit': 'sdk-v2-1.29.0'}, 'properties': {}, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-test-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-test-ws', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x000001BEBAD73C40>, 'display_name': 'azure-ml-test-ws', 'location': 'westeurope', 'resource_g

In [13]:
valid_workspace_name

'azure-ml-test-ws'

In [14]:
from azure.ai.ml.entities import Workspace
from utils.consts import AZUREML_RESOURCE_LOCATION as resource_group_location, AZUREML_SUBSCRIPTION_ID as subscription_id, AZUREML_RESOURCE_GROUP as resource_group, AZUREML_WORKSPACE_NAME as workspace_name

valid_workspace_name = workspace_name.replace("dev", "valid")
valid_resource_group = resource_group.replace("dev", "test")

azureml_valid_ws = Workspace(
    name=valid_workspace_name,
    location=resource_group_location,
    storage_account=azureml_test_ws.storage_account,
    key_vault=azureml_test_ws.key_vault,
    application_insights=azureml_test_ws.application_insights,
)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=valid_resource_group,
)

ml_client.workspaces.begin_create(azureml_valid_ws).result()

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
The deployment request azure-ml-valid-ws-7386103 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2Fa1267753-4c98-48c1-a8e9-9c7169202ffd%2FresourceGroups%2Fazure-ml-test-rg%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Fazure-ml-valid-ws-7386103
Creating AzureML Workspace: (azure-ml-valid-ws  ) .............................................................  Done (5m 27s)
Total time : 5m 29s



Workspace({'kind': 'default', 'print_as_yaml': False, 'discovery_url': 'https://westeurope.api.azureml.ms/discovery', 'mlflow_tracking_uri': 'azureml://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-test-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-valid-ws', 'workspace_id': '64184d19-9511-4665-8f62-72ce2c90d69e', 'feature_store_settings': None, 'name': 'azure-ml-valid-ws', 'description': 'azure-ml-valid-ws', 'tags': {'createdByToolkit': 'sdk-v2-1.29.0'}, 'properties': {}, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-test-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-valid-ws', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x000001BEEE8A0910>, 'display_name': 'azure-ml-valid-ws', 'location': 'westeurope', 'resou

## Manage Data

### Datastores

In [8]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100
workspaceworkingdirectory
workspaceblobstore
workspaceartifactstore
workspacefilestore


#### Create a datastore

You can create a new datastore in a default azure ml storage account container. However you can also create a new container in an existing storage account and register it as a datastore. You can do it in azure portal or programmatically as shown below.

##### Create a data container in existing storage account

In [4]:
from azure.storage.blob import BlobServiceClient
from utils.consts import AZUREML_STORAGE_ACCOUNT_NAME, AZUREML_STORAGE_ACCOUNT_ACCESS_KEY

storage_container_name = "dmdp100-azureml"

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={AZUREML_STORAGE_ACCOUNT_NAME};"
    f"AccountKey={AZUREML_STORAGE_ACCOUNT_ACCESS_KEY};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmdp100-azureml' created.


##### Create a datastore

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from utils.consts import AZUREML_STORAGE_ACCOUNT_NAME, AZUREML_STORAGE_ACCOUNT_ACCESS_KEY

datastore_name = "dmdp100"
storage_container_name = "dmdp100-azureml"

store = AzureBlobDatastore(
    name=datastore_name, # name cannot contain "-"
    description="Blob Storage for DP-100 certification prep",
    account_name=AZUREML_STORAGE_ACCOUNT_NAME,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=AZUREML_STORAGE_ACCOUNT_ACCESS_KEY
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/datastores/dmdp100', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7a5cf8b82500>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmdp100-azureml', 'account_name': 'polandaidevmlst', 'endpoint': 'core.windows.net', 'protocol': 'https'})

##### (Optional) Set a datastore as default


In [27]:
print(f"Current default datastore: {ml_client.datastores.get_default().name}")

Current default datastore: workspaceblobstore


In [ ]:
# Doesn't work
# from utils.consts import AZUREML_WORKSPACE_NAME
# datastore_name = "dmdp100"
# workspace = ml_client.workspaces.get(AZUREML_WORKSPACE_NAME)
# workspace.default_datastore = store
# ml_client.workspaces.begin_update(workspace).result()
# print(f"Default datastore: {ml_client.datastores.get_default().name}")

Workspace({'kind': 'default', 'print_as_yaml': False, 'discovery_url': 'https://westeurope.api.azureml.ms/discovery', 'mlflow_tracking_uri': 'azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw', 'workspace_id': '98a4e2ed-9e4a-44b6-a65e-e3273c9dc09b', 'feature_store_settings': None, 'name': 'polandaidevml-mlw', 'description': '', 'tags': {}, 'properties': {}, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x7a5ce0a45d50>, 'display_name': '', 'location': 'westeurope', 'resource_group': 'polan

### Data Assets

#### Creating Data Assets

In [30]:
datastore_name = "dmdp100"

##### URI_FILE

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/azure-ml-labs-data/diabetes/diabetes.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="diabetes-data-file",
)

ml_client.data.create_or_update(my_data)

Uploading diabetes.csv (< 1 MB): 100%|██████████| 518k/518k [00:00<00:00, 17.6MB/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/3d8efe6c7dafbd432031e3f030cc92dd5acd9da03214f6d3ac06b02ed81cc551/diabetes.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-file', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/diabetes-data-file/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <az

In [31]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file-raw"
)

ml_client.data.create_or_update(my_data)

Uploading telco-customer-churn.csv (< 1 MB): 100%|██████████| 970k/970k [00:00<00:00, 28.8MB/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/c9f74bf3dd2417bba280f0ceef276024cbe95a3935ed06a94a7f357d15495775/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file-raw', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-file-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creati

##### URI_FOLDER

In [32]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder-raw",
)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|██████████| 970588/970588 [00:00<00:00, 19806944.60it/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/6b41ae31c618bc8c542df493b94c95d2c0ad3f6843ff934bcd1b6294be86382e/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder-raw', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-folder-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml.e

##### MLTABLE

In [33]:
%%writefile data/azure-ml-labs-data/diabetes/MLTable

paths:
  - file: ./diabetes.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/azure-ml-labs-data/diabetes/MLTable


In [35]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/azure-ml-labs-data/diabetes'

my_data = Data(
    path=data_path,
    datastore=datastore_name,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes.csv in data folder",
    name="diabetes-data-table",
)

ml_client.data.create_or_update(my_data)

Uploading diabetes (0.52 MBs): 100%|██████████| 517871/517871 [00:00<00:00, 8885768.78it/s]




Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/60aba111c79a4a33c719346aff233bf95950caa2dccbc2f200fe25c59119fad6/diabetes/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./diabetes.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes-data-table', 'description': 'MLTable pointing to diabetes.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/diabetes-data-table/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_context': <azure.ai.ml.entities._syste

In [36]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/telco-churn-data/MLTable


In [37]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    datastore=datastore_name,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table-raw",

)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/workspaces/polandaidevml-mlw/datastores/dmdp100/paths/LocalUpload/6b41ae31c618bc8c542df493b94c95d2c0ad3f6843ff934bcd1b6294be86382e/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table-raw', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/data/telco-churn-table-raw/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn', 'creation_co

#### Read the Data Assets Locally

In [2]:
import pandas as pd
import mltable

In [3]:
data_asset = ml_client.data.get("telco-churn-file-raw", version="1")
df = pd.read_csv(data_asset.path)
df.sample(2)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1705,4918-FYJNT,Female,1,Yes,No,55,Yes,Yes,Fiber optic,No,...,Yes,Yes,No,No,Month-to-month,No,Electronic check,90.45,5044.8,No
1241,0096-FCPUF,Male,0,No,No,30,Yes,Yes,DSL,Yes,...,No,No,No,Yes,Month-to-month,Yes,Mailed check,64.50,1888.45,No


In [4]:
data_asset = ml_client.data.get("telco-churn-folder-raw", version="1")
path = {
  'folder': data_asset.path
}
# tbl = mltable.from_delimited_files(paths=[path])
# df = tbl.to_pandas_dataframe()
df = pd.read_csv(os.path.join(data_asset.path, 'telco-customer-churn.csv'))
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
890,5898-IGSLP,Male,0,Yes,Yes,31,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,No,No,Month-to-month,No,Electronic check,89.3,2823,No
3380,5178-LMXOP,Male,1,Yes,No,1,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.1,95.1,Yes


In [5]:
data_asset = ml_client.data.get("diabetes-data-table", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


In [6]:
data_asset = ml_client.data.get("telco-churn-table-raw", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,...,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,...,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,...,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,...,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


## Manage Compute and Environments

## Setting up an Environment

In [16]:
env_file_path = "environment/azure-ml-environment/dmdp100env.yml"
main_env_name = "dmdp100env"
parent_image = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env_content = f"""
channels:
  - conda-forge
dependencies:
  - python=3.10.11
  - pip=22.3.1
  - pip:
      - scipy
      - pandas
      - scikit-learn
      - adlfs
      - fsspec
      - xgboost
      - lightgbm
      - mlflow<3.0.0
      - azureml-mlflow
      - matplotlib
      - tqdm
      - seaborn
name: {main_env_name}
"""

with open(env_file_path, "w") as f:
    f.write(env_content)


In [17]:
from azure.ai.ml.entities import Environment

env_docker_conda = Environment(
    image=parent_image,
    conda_file=env_file_path,
    name=main_env_name,
    description="Environment created for main dp100 prep.",
)
ml_client.environments.create_or_update(env_docker_conda)
# NOTE: TO create an environment image using a compute cluster you have to have a premium Container Registry and update your workspace accordingly:
# ws.image_build_compute = "your-cluster"
# ml_client.workspaces.begin_update(ws)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'dmdp100env', 'description': 'Environment created for main dp100 prep.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/azure-ml-dev-rg/providers/Microsoft.MachineLearningServices/workspaces/azure-ml-dev-ws/environments/dmdp100env/versions/1', 'Resource__source_path': '', 'base_path': 'c:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001BEEE872EC0>, 'serialize': <msrest.serialization.Serializer object at 0x000001BEEE873BE0>, 'version': '1', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.10.11', 'pip=22.3.1', {'pip': ['s

## Manage Compute

### Create a Compute Cluster

#### CPU Cluster

In [8]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "dmdp100-cpu-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="STANDARD_DS11_V2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=120,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)


You already have a cluster named dmdp100-cpu-cluster, we'll reuse it as is.


#### GPU Cluster

In [3]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
gpu_compute_target = "dmdp100-gpu-cluster"
# try:
#     # let's see if the compute target already exists
#     gpu_cluster = ml_client.compute.get(gpu_compute_target)
#     print(
#         f"You already have a cluster named {gpu_compute_target}, we'll reuse it as is."
#     )

# except Exception:
print("Creating a new gpu compute target...")

# Let's create the Azure ML compute object with the intended parameters
gpu_cluster = AmlCompute(
    name=gpu_compute_target,
    # Azure ML Compute is the on-demand VM service
    type="amlcompute",
    # VM Family
    size="STANDARD_NC4AS_T4_V3",
    # Minimum running nodes when there is no job running
    min_instances=0,
    # Nodes in cluster
    max_instances=4,
    # How many seconds will the node running after the job termination
    idle_time_before_scale_down=120,
    # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
    tier="LowPriority",
)

# Now, we pass the object to MLClient's create_or_update method
gpu_cluster = ml_client.compute.begin_create_or_update(gpu_cluster)


Creating a new gpu compute target...


### Create a Compute Instance

In [11]:
# Compute Instances need to have a unique name across the region.
# Here we create a unique name with current datetime
from azure.ai.ml.entities import ComputeInstance
import datetime

ci_basic_name = "dmdp100ci" + datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_basic_name = ci_basic_name[:24]
ci_basic = ComputeInstance(name=ci_basic_name, size="STANDARD_DS11_V2", idle_time_before_shutdown_minutes="15")
ml_client.begin_create_or_update(ci_basic).result()

ComputeInstance({'state': 'Running', 'last_operation': {'operation_name': 'Create', 'operation_time': '2025-11-03T17:29:37.478Z', 'operation_status': 'Succeeded', 'operation_trigger': 'User'}, 'os_image_metadata': <azure.ai.ml.entities._compute._image_metadata.ImageMetadata object at 0x7ed995916ad0>, 'services': [{'display_name': 'Jupyter', 'endpoint_uri': 'https://dmdp100ci202511031729.westeurope.instances.azureml.ms/tree/'}, {'display_name': 'Jupyter Lab', 'endpoint_uri': 'https://dmdp100ci202511031729.westeurope.instances.azureml.ms/lab'}], 'type': 'computeinstance', 'created_on': '2025-11-03T17:29:29.822327+0000', 'provisioning_state': 'Succeeded', 'provisioning_errors': None, 'name': 'dmdp100ci202511031729', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/computes/dmdp100ci202511031729', 'Re